# Step 1: Data Ingestion from Stooq API


In [3]:
import pandas as pd
import requests
import io

# 1. Our Asset Universe (6 Large-Cap Tech Stocks)
# US stocks in the Stooq system are generally called with a .US extension
tickers = ['AAPL.US', 'MSFT.US', 'AMZN.US', 'GOOGL.US', 'TSLA.US', 'NVDA.US']

# Creating an empty dictionary to store the raw data
raw_data = {}


for ticker in tickers:
    # Stooq API URL format specified by the user
    url = f"https://stooq.com/q/d/l/?s={ticker}&i=d"

    # Sending a request to the URL
    response = requests.get(url)

    # If the request is successful (returns HTTP 200 status code), proceed
    if response.status_code == 200:
        # Converting the received CSV formatted text into a Pandas DataFrame
        df = pd.read_csv(io.StringIO(response.text))

        # Saving the DataFrame to our dictionary
        raw_data[ticker] = df
        print(f"[SUCCESS] {ticker} data fetched. Total rows (days): {len(df)}")
    else:
        print(f"[ERROR] Could not fetch {ticker} data. Status Code: {response.status_code}")

print("\nAll data fetching operations completed!")

# Let's look at the most recent 5 rows for Apple (AAPL) as an example
# Stooq might provide data from oldest to newest or vice versa, we check the end with tail()
raw_data['AAPL.US'].tail()

[SUCCESS] AAPL.US data fetched. Total rows (days): 10452
[SUCCESS] MSFT.US data fetched. Total rows (days): 10069
[SUCCESS] AMZN.US data fetched. Total rows (days): 7239
[SUCCESS] GOOGL.US data fetched. Total rows (days): 5419
[SUCCESS] TSLA.US data fetched. Total rows (days): 3944
[SUCCESS] NVDA.US data fetched. Total rows (days): 6819

All data fetching operations completed!


,Date,Open,High,Low,Close,Volume
10447,2026-02-26,274.945,276.11,270.795,272.950,32191569
10448,2026-02-27,272.810,272.81,262.890,264.180,72232024
10449,2026-03-02,262.410,266.53,260.200,264.720,41827946
10450,2026-03-03,263.480,265.56,260.130,263.690,38568921
10451,2026-03-04,264.650,266.15,261.420,265.095,7387275


# Step 2: Data Preprocessing & Cleaning


In [4]:
import pandas as pd

# Dictionary to hold the new, cleaned data
close_prices = {}

print("Merging and cleaning data...")

for ticker, df in raw_data.items():
    # Converting the Date column to datetime format
    df['Date'] = pd.to_datetime(df['Date'])

    # Setting the Date column as the main index of the table
    df.set_index('Date', inplace=True)

    # Removing the '.US' part from the ticker name (e.g., AAPL.US -> AAPL)
    clean_ticker = ticker.split('.')[0]

    # Extracting only 'Close' prices and assigning them to the dictionary
    close_prices[clean_ticker] = df['Close']

# Concatenating the closing prices of all stocks side-by-side into a single table
portfolio_df = pd.concat(close_prices, axis=1)

# Step 2: Filtering the last 5 years of data
# End date is the most recent date in the dataset
end_date = portfolio_df.index.max()
# Start date is exactly 5 years prior
start_date = end_date - pd.DateOffset(years=5)

# Slicing the table according to this 5-year interval
portfolio_df = portfolio_df.loc[start_date:end_date]

# Step 3: Cleaning missing data
# First, carrying forward the previous day's price (ffill)
# If there is missing data on the very first day, filling it backward (bfill)
portfolio_df = portfolio_df.ffill().bfill()

print("\n[SUCCESS] Data cleaning completed!")
print(f"Start Date: {portfolio_df.index.min().date()}")
print(f"End Date: {portfolio_df.index.max().date()}")
print(f"Total Trading Days: {len(portfolio_df)}")
print("\nCleaned Final Table (First 5 Rows):")

# Displaying the first 5 rows of the table
portfolio_df.head()

Merging and cleaning data...

[SUCCESS] Data cleaning completed!
Start Date: 2021-03-04
End Date: 2026-03-04
Total Trading Days: 1256

Cleaned Final Table (First 5 Rows):


,AAPL,MSFT,AMZN,GOOGL,TSLA,NVDA
Date,,,,,,
2021-03-04,117.121,219.666,148.878,101.328,207.147,12.3384
2021-03-05,118.367,224.386,150.023,104.475,199.317,12.4294
2021-03-08,113.437,220.312,147.598,100.012,187.667,11.5638
2021-03-09,118.045,226.486,153.142,101.650,224.527,12.4924
2021-03-10,116.983,225.176,152.882,101.442,222.687,12.4404


#Step 3: Exporting Cleaned Data

In [6]:
from google.colab import files

# Converting the data into a CSV file
filename = "YAP471_Cleaned_Close_Prices.csv"
portfolio_df.to_csv(filename)

print(f"[SUCCESS] Table saved into Colab as {filename}.")
print("The file is now being downloaded to your computer...")

# Command to download the CSV file to your computer
files.download(filename)

[SUCCESS] Table saved into Colab as YAP471_Cleaned_Close_Prices.csv.
The file is now being downloaded to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>